In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\75\AppData\Local\Temp\ipykernel_2292\3336491994.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\75\AppData\Local\Temp\ipykernel_2292\3336491994.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):


In [2]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 1.加载系统环境变量
load_dotenv()

# 2.定义模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 3.定义工具 - MCP工具
# 3.1 创建客户端
client = MultiServerMCPClient(
    # 基于标准MCP配置，去掉第一层，然后增加传输协议
    {
        "time-mcp": {
            "transport":"stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },
        "12306-mcp": {
        "transport": "streamable_http",
        "url": "https://mcp.api-inference.modelscope.net/fd9cf65604c24e/mcp"
    }

    }
)
# 3.2 获取工具列表
tools = await client.get_tools()

# 4.定义智能体
agent = create_agent(
    model=model,
    tools=tools
)

# 5.异步调用
response = await agent.ainvoke({
    # "messages":[
    #     HumanMessage(content="当前服务器的时间是多少?")
    # ]
    "messages":[
        HumanMessage(content="今天北京到广州")
    ]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

当前服务器的时间是多少?
================================== Ai Message ==================================
Tool Calls:
  current_time (call_00_OCjlxWf6W4eexR4NMVxF9918)
 Call ID: call_00_OCjlxWf6W4eexR4NMVxF9918
  Args:
    format: YYYY-MM-DD HH:mm:ss
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': 'Current UTC time is 2026-10-19 01:43:05, and the time in Asia/Shanghai is 2026-10-19 09:43:05.', 'id': 'lc_f51663ba-84ff-4393-bfd6-c3e4598f451d'}]
================================== Ai Message ==================================

当前服务器时间（UTC）为 **2026-10-19 01:43:05**，对应北京时间（Asia/Shanghai）为 **2026-10-19 09:43:05**。
